# 🍌 ระบบจำแนกระดับความสุกของกล้วยจากภาพถ่าย ด้วยค่าสี HSV และ Logistic Regression
**วิชา AI for EE (67342EPE1) — Mini Project**

ขั้นตอน: ภาพกล้วย → ตัดฉากหลัง → ดึงค่าสี HSV → Logistic Regression → ผลระดับความสุก → วัดผล

> วิธีใช้: กด **Runtime → Run all** (หรือกด ▶ ทีละเซลล์จากบนลงล่าง)

## 1. ตั้งค่า (แก้ได้ตรงนี้)

In [ ]:
# ===== ตั้งค่า =====
DRIVE_FOLDER = '/content/drive/MyDrive/BananaAI'   # โฟลเดอร์บน Google Drive ที่เก็บไฟล์ zip
MERGE_SEMI   = False   # False = 4 ระดับ (Green / Semi-ripe / Ripe / Overripe)
                       # True  = 3 ระดับ (รวม Semi-ripe เข้ากับ Ripe → ดิบ / สุก / งอม)
TEST_SIZE    = 0.2     # แบ่งข้อมูลไว้ทดสอบ 20%
SEED         = 42      # ค่าสุ่มคงที่ จะได้ผลเหมือนเดิมทุกครั้งที่รัน

## 2. เชื่อม Google Drive และแตกไฟล์ dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import glob, os, zipfile
zips = glob.glob(os.path.join(DRIVE_FOLDER, '*.zip'))
print('เจอไฟล์ zip:', zips)
for z in zips:
    zipfile.ZipFile(z).extractall('/content/data')
print('แตกไฟล์เสร็จ')

In [ ]:
import glob, os
# หาโฟลเดอร์ที่มีโฟลเดอร์ย่อย Green / Ripe / Overripe อยู่ข้างใน (ไม่ว่าจะซ้อนกี่ชั้น)
cands = [os.path.dirname(p) for p in glob.glob('/content/data/**/Green', recursive=True)]
assert cands, 'ไม่เจอโฟลเดอร์ Green — เช็กว่าอัป zip ถูกไฟล์ไหม'
DATA_ROOT = cands[0]
print('DATA_ROOT =', DATA_ROOT)
print(sorted(os.listdir(DATA_ROOT)))

## 3. โหลดรายชื่อรูปและป้ายกำกับ (label)

In [ ]:
import glob, os, numpy as np, cv2, matplotlib.pyplot as plt

CLASS_FOLDERS = ['Green', 'Semi-ripe', 'Ripe', 'Overripe']
if MERGE_SEMI:
    CLASS_NAMES = ['Unripe', 'Ripe', 'Overripe']
    FOLDER_TO_LABEL = {'Green': 0, 'Semi-ripe': 1, 'Ripe': 1, 'Overripe': 2}
else:
    CLASS_NAMES = ['Green', 'Semi-ripe', 'Ripe', 'Overripe']
    FOLDER_TO_LABEL = {c: i for i, c in enumerate(CLASS_FOLDERS)}

paths, labels = [], []
for folder in CLASS_FOLDERS:
    for f in sorted(glob.glob(os.path.join(DATA_ROOT, folder, '*'))):
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):   # รองรับ .JPG ตัวใหญ่ด้วย
            paths.append(f); labels.append(FOLDER_TO_LABEL[folder])
labels = np.array(labels)
print('จำนวนรูปทั้งหมด:', len(paths))
for i, n in enumerate(CLASS_NAMES):
    print(f'  {n:10s}: {(labels == i).sum()} รูป')

In [ ]:
# ดูรูปตัวอย่างแต่ละระดับ
fig, ax = plt.subplots(len(CLASS_FOLDERS), 5, figsize=(12, 2.6 * len(CLASS_FOLDERS)))
for r, folder in enumerate(CLASS_FOLDERS):
    fs = sorted(glob.glob(os.path.join(DATA_ROOT, folder, '*')))[:5]
    for c, f in enumerate(fs):
        ax[r, c].imshow(cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB)); ax[r, c].axis('off')
    ax[r, 0].set_title(folder, loc='left')
plt.tight_layout(); plt.show()

## 4. ประมวลผลภาพ: แปลงเป็น HSV และตัดฉากหลัง
- **H (Hue)** = เฉดสี (เขียว / เหลือง / น้ำตาล) **S (Saturation)** = ความสด **V (Value)** = ความสว่าง
- ฉากหลังเป็นสีขาว/เทา → ค่า S ต่ำ ส่วนกล้วยมีค่า S สูง หรือถ้างอมจนดำก็มีค่า V ต่ำ
- เลือกเฉพาะพิกเซลที่ `S > 50` หรือ `V < 90` เป็น "ตัวกล้วย" แล้วใช้ Morphology ลบจุดรบกวน

In [ ]:
IMG_SIZE = 256

def banana_mask(hsv):
    h, s, v = cv2.split(hsv)
    mask = ((s > 50) | (v < 90)).astype(np.uint8) * 255
    k = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k)    # ลบจุดเล็กๆ ที่ฉากหลัง
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)   # อุดรูในตัวกล้วย
    return mask

# แสดงผลการตัดฉากหลัง
fig, ax = plt.subplots(2, len(CLASS_FOLDERS), figsize=(12, 6))
for i, folder in enumerate(CLASS_FOLDERS):
    f = sorted(glob.glob(os.path.join(DATA_ROOT, folder, '*')))[0]
    img = cv2.resize(cv2.imread(f), (IMG_SIZE, IMG_SIZE))
    m = banana_mask(cv2.cvtColor(img, cv2.COLOR_BGR2HSV))
    ax[0, i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax[0, i].set_title(folder); ax[0, i].axis('off')
    ax[1, i].imshow(cv2.cvtColor(cv2.bitwise_and(img, img, mask=m), cv2.COLOR_BGR2RGB)); ax[1, i].set_title('mask'); ax[1, i].axis('off')
plt.tight_layout(); plt.show()

## 5. สกัดคุณลักษณะ (Feature Extraction)
แต่ละรูปเหลือตัวเลข **6 ค่า** = ค่าเฉลี่ย (mean) และส่วนเบี่ยงเบนมาตรฐาน (std) ของ H, S, V เฉพาะบริเวณตัวกล้วย
- mean บอก "สีโดยรวม" ส่วน std บอก "สีสม่ำเสมอแค่ไหน" (กล้วยงอมมีจุดดำ ทำให้ std สูง)

In [ ]:
FEATURE_NAMES = ['H_mean', 'S_mean', 'V_mean', 'H_std', 'S_std', 'V_std']

def extract_features(img_bgr):
    img = cv2.resize(img_bgr, (IMG_SIZE, IMG_SIZE))
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    m = banana_mask(hsv)
    px = hsv[m > 0].astype(float)
    if len(px) < 100:                 # กันกรณีตัดฉากหลังไม่สำเร็จ ให้ใช้ทั้งภาพ
        px = hsv.reshape(-1, 3).astype(float)
    return np.concatenate([px.mean(axis=0), px.std(axis=0)])

X = np.array([extract_features(cv2.imread(p)) for p in paths])
y = labels
print('X shape =', X.shape, '(จำนวนรูป, จำนวนฟีเจอร์)')

import pandas as pd
df = pd.DataFrame(X, columns=FEATURE_NAMES); df['class'] = [CLASS_NAMES[i] for i in y]
df.groupby('class', sort=False)[FEATURE_NAMES].mean().round(1)

In [ ]:
# กราฟกระจาย H_mean vs V_mean — ดูว่าแต่ละระดับแยกกันด้วยสีได้แค่ไหน
plt.figure(figsize=(7, 5))
for i, n in enumerate(CLASS_NAMES):
    plt.scatter(X[y == i, 0], X[y == i, 2], s=12, alpha=0.6, label=n)
plt.xlabel('Hue mean'); plt.ylabel('Value (brightness) mean'); plt.legend(); plt.title('HSV features by ripeness')
plt.grid(alpha=0.3); plt.show()

## 6. แบ่งข้อมูล และเทรนโมเดล Logistic Regression
- แบ่ง **Train 80% / Test 20%** แบบ stratified (สัดส่วนแต่ละคลาสเท่าเดิม)
- **StandardScaler** ปรับให้ทุกฟีเจอร์อยู่ในสเกลเดียวกัน (H มีค่า 0–180 แต่ std มีค่าเล็ก)
- **Logistic Regression** แบบหลายคลาส: คำนวณ z = w·x + b ของแต่ละคลาส แล้วแปลงเป็นความน่าจะเป็นด้วย Softmax → เลือกคลาสที่ความน่าจะเป็นสูงสุด

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED)
print('Train:', len(X_train), 'รูป | Test:', len(X_test), 'รูป')

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
model.fit(X_train, y_train)
print('เทรนเสร็จ ✅')

## 7. วัดผล (Evaluation)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

y_pred = model.predict(X_test)
print(f'Train accuracy: {model.score(X_train, y_train)*100:.2f}%')
print(f'Test accuracy : {accuracy_score(y_test, y_pred)*100:.2f}%\n')
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=3))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=CLASS_NAMES, cmap='YlGn')
plt.title('Confusion Matrix (Test set)'); plt.show()

In [ ]:
# ตรวจซ้ำด้วย 5-Fold Cross Validation (ผลน่าเชื่อถือกว่าการแบ่งครั้งเดียว)
from sklearn.model_selection import cross_val_score, StratifiedKFold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scores = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), X, y, cv=cv)
print('Accuracy แต่ละ fold:', (scores*100).round(2))
print(f'เฉลี่ย = {scores.mean()*100:.2f}% ± {scores.std()*100:.2f}%')

In [ ]:
# เปรียบเทียบ: ใช้แค่ค่าเฉลี่ย H,S,V (3 ฟีเจอร์) vs mean+std (6 ฟีเจอร์)
for name, cols in [('mean HSV (3 features)', [0, 1, 2]), ('mean+std HSV (6 features)', list(range(6)))]:
    s = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), X[:, cols], y, cv=cv)
    print(f'{name:28s}: {s.mean()*100:.2f}%')

In [ ]:
# ดูรูปที่โมเดลทายผิด
idx_test = train_test_split(np.arange(len(paths)), test_size=TEST_SIZE, stratify=y, random_state=SEED)[1]
wrong = [i for i, t, p in zip(idx_test, y_test, y_pred) if t != p][:8]
if wrong:
    fig, ax = plt.subplots(1, len(wrong), figsize=(2.5 * len(wrong), 3))
    ax = np.atleast_1d(ax)
    for a, i in zip(ax, wrong):
        a.imshow(cv2.cvtColor(cv2.imread(paths[i]), cv2.COLOR_BGR2RGB)); a.axis('off')
        a.set_title(f'true: {CLASS_NAMES[y[i]]}\npred: {CLASS_NAMES[model.predict(X[i:i+1])[0]]}', fontsize=9)
    plt.tight_layout(); plt.show()
else:
    print('ไม่มีรูปที่ทายผิด 🎉')

## 8. เดโม: อัปโหลดรูปกล้วยของตัวเองแล้วให้ AI ทาย

In [ ]:
from google.colab import files
uploaded = files.upload()
for name, data in uploaded.items():
    img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
    prob = model.predict_proba([extract_features(img)])[0]
    pred = CLASS_NAMES[prob.argmax()]
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.axis('off')
    plt.title(f'{pred} ({prob.max()*100:.1f}%)'); plt.show()
    for n, p in zip(CLASS_NAMES, prob):
        print(f'  {n:10s}: {p*100:5.1f}%')

## 9. บันทึกโมเดลไว้ใน Google Drive

In [ ]:
import joblib
joblib.dump({'model': model, 'class_names': CLASS_NAMES}, os.path.join(DRIVE_FOLDER, 'banana_model.joblib'))
print('บันทึกแล้วที่', os.path.join(DRIVE_FOLDER, 'banana_model.joblib'))